In [1]:
import sqlite3
from datetime import datetime, timezone
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
DB_PATH = PROJECT_ROOT / "data" / "db" / "retail.db"
REPORTS_DIR = PROJECT_ROOT / "reports"
MODELS_DIR = PROJECT_ROOT / "models"

SEED = 42
FEATURE_COLUMNS = ["day_of_week", "is_weekend", "month", "day_of_month", "is_festive_season",
                   "store_type_code", "sales_lag_7", "sales_lag_14", "sales_lag_28",
                   "sales_mean_7d", "sales_mean_28d"]

conn = sqlite3.connect(DB_PATH)
data = pd.read_sql("SELECT * FROM features_daily_sales", conn, parse_dates=["date"])

print(data.shape)
print("From", data["date"].min().date(), "to", data["date"].max().date())

(8364, 17)
From 2024-02-04 to 2025-12-31


In [2]:
TRAIN_START = pd.Timestamp("2024-07-01")
TEST_START = pd.Timestamp("2025-09-01")

train = data[(data["date"] >= TRAIN_START) & (data["date"] < TEST_START)]
test = data[data["date"] >= TEST_START]

print(f"Train: {len(train):,} rows, {train['date'].min().date()} to {train['date'].max().date()}")
print(f"Test:  {len(test):,} rows, {test['date'].min().date()} to {test['date'].max().date()}")
assert train["date"].max() < test["date"].min(), "train and test overlap in time"

Train: 5,124 rows, 2024-07-01 to 2025-08-31
Test:  1,464 rows, 2025-09-01 to 2025-12-31


In [3]:
def evaluate(actual, predicted):
    error = actual - predicted
    return {
        "MAE": np.abs(error).mean(),
        "RMSE": np.sqrt((error ** 2).mean()),
        "WAPE_pct": 100 * np.abs(error).sum() / np.abs(actual).sum(),
    }

baseline_pred = test["sales_lag_7"]
results = {"seasonal_naive": evaluate(test["sales"], baseline_pred)}

pd.DataFrame(results).T.round(1)

,MAE,RMSE,WAPE_pct
seasonal_naive,49607.8,83480.5,44.5


In [4]:
model = lgb.LGBMRegressor(
    n_estimators=400,
    learning_rate=0.03,
    num_leaves=15,
    min_child_samples=20,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    random_state=SEED,
    verbose=-1,
)

model.fit(train[FEATURE_COLUMNS], train["sales"])

lgb_pred = model.predict(test[FEATURE_COLUMNS])
results["lightgbm"] = evaluate(test["sales"], lgb_pred)

pd.DataFrame(results).T.round(1)

,MAE,RMSE,WAPE_pct
seasonal_naive,49607.8,83480.5,44.5
lightgbm,37449.5,62114.8,33.6


In [5]:
test = test.copy()
test["pred_baseline"] = baseline_pred
test["pred_lightgbm"] = lgb_pred
test["month_name"] = test["date"].dt.strftime("%Y-%m")

rows = []
for segment_col in ["store_type", "region", "month_name"]:
    for segment_value, group in test.groupby(segment_col):
        base = evaluate(group["sales"], group["pred_baseline"])
        lgbm = evaluate(group["sales"], group["pred_lightgbm"])
        rows.append({
            "segment_type": segment_col,
            "segment": segment_value,
            "days": len(group),
            "baseline_WAPE_pct": round(base["WAPE_pct"], 1),
            "lightgbm_WAPE_pct": round(lgbm["WAPE_pct"], 1),
        })

segments = pd.DataFrame(rows)
segments["improvement_pts"] = segments["baseline_WAPE_pct"] - segments["lightgbm_WAPE_pct"]
segments

,segment_type,segment,days,baseline_WAPE_pct,lightgbm_WAPE_pct,improvement_pts
0,store_type,Express,732,50.8,38.4,12.4
1,store_type,Flagship,244,37.0,27.2,9.8
2,store_type,Standard,488,43.7,33.5,10.2
3,region,East,366,45.3,34.0,11.3
4,region,North,366,45.4,34.2,11.2
5,region,South,366,48.9,38.7,10.2
6,region,West,366,40.0,29.1,10.9
7,month_name,2025-09,360,50.0,35.6,14.4
8,month_name,2025-10,372,42.6,31.2,11.4
9,month_name,2025-11,360,39.9,31.0,8.9


In [6]:
importance = pd.DataFrame({
    "feature": FEATURE_COLUMNS,
    "importance_gain": model.booster_.feature_importance(importance_type="gain"),
})
importance["importance_pct"] = (100 * importance["importance_gain"] / importance["importance_gain"].sum()).round(1)
importance = importance.sort_values("importance_pct", ascending=False).drop(columns="importance_gain")

importance

,feature,importance_pct
10,sales_mean_28d,43.1
9,sales_mean_7d,14.6
0,day_of_week,10.4
5,store_type_code,7.8
6,sales_lag_7,5.0
8,sales_lag_28,4.6
7,sales_lag_14,4.6
2,month,3.2
3,day_of_month,2.8
1,is_weekend,2.5


In [7]:
MODEL_VERSION = "forecast_lgbm_v1"
predicted_at = datetime.now(timezone.utc).isoformat(timespec="seconds")

predictions = test[["store_id", "date", "store_type", "region", "sales",
                    "pred_baseline", "pred_lightgbm"]].rename(columns={"sales": "actual_sales"})
predictions["date"] = predictions["date"].dt.strftime("%Y-%m-%d")
predictions["pred_baseline"] = predictions["pred_baseline"].round(2)
predictions["pred_lightgbm"] = predictions["pred_lightgbm"].round(2)
predictions["model_version"] = MODEL_VERSION
predictions["predicted_at"] = predicted_at

evaluation = pd.DataFrame(results).T.round(2).reset_index().rename(columns={"index": "model"})
evaluation["model_version"] = ["seasonal_naive_v1", MODEL_VERSION]
evaluation["train_period"] = f"{TRAIN_START.date()} to {train['date'].max().date()}"
evaluation["test_period"] = f"{TEST_START.date()} to {test['date'].max().date()}"
evaluation["evaluated_at"] = predicted_at

tables = {
    "forecast_predictions": predictions,
    "forecast_evaluation": evaluation,
    "forecast_segment_errors": segments.drop(columns="days"),
    "forecast_feature_importance": importance,
}
for name, df in tables.items():
    df.to_sql(name, conn, if_exists="replace", index=False)
    df.to_csv(REPORTS_DIR / f"{name}.csv", index=False)
    print(f"Saved {name}: {len(df)} rows")

MODELS_DIR.mkdir(exist_ok=True)
model.booster_.save_model(str(MODELS_DIR / f"{MODEL_VERSION}.txt"))
print("Saved model file:", MODELS_DIR / f"{MODEL_VERSION}.txt")

evaluation

Saved forecast_predictions: 1464 rows
Saved forecast_evaluation: 2 rows
Saved forecast_segment_errors: 11 rows
Saved forecast_feature_importance: 11 rows
Saved model file: C:\Users\samee\Desktop\retail-project\models\forecast_lgbm_v1.txt


,model,MAE,RMSE,WAPE_pct,model_version,train_period,test_period,evaluated_at
0,seasonal_naive,49607.79,83480.51,44.52,seasonal_naive_v1,2024-07-01 to 2025-08-31,2025-09-01 to 2025-12-31,2026-11-02T09:33:56+00:00
1,lightgbm,37449.47,62114.75,33.61,forecast_lgbm_v1,2024-07-01 to 2025-08-31,2025-09-01 to 2025-12-31,2026-11-02T09:33:56+00:00


In [8]:
limitations = pd.DataFrame({"limitation": [
    "Daily store-level sales are noisy: even the better model is off by about 34% on a typical day. "
    "Weekly totals would be much more accurate if the business only needs weekly planning.",
    "Small (Express) stores have higher error (38%) than Flagships (27%) because a few large orders swing their totals.",
    "Only one previous festive season (2024) was available for training, so festive forecasts rest on limited history.",
    "Forecasts are of gross sales; about 3.9% of item revenue is later returned.",
    "The model assumes the future behaves like the past: new stores, promotions or price changes are not known to it.",
    "The data is synthetic; accuracy on real retail data would need to be re-measured.",
]})
limitations.to_sql("forecast_limitations", conn, if_exists="replace", index=False)
limitations.to_csv(REPORTS_DIR / "forecast_limitations.csv", index=False)

pd.set_option("display.max_colwidth", None)
limitations

,limitation
0,Daily store-level sales are noisy: even the better model is off by about 34% on a typical day. Weekly totals would be much more accurate if the business only needs weekly planning.
1,Small (Express) stores have higher error (38%) than Flagships (27%) because a few large orders swing their totals.
2,"Only one previous festive season (2024) was available for training, so festive forecasts rest on limited history."
3,Forecasts are of gross sales; about 3.9% of item revenue is later returned.
4,"The model assumes the future behaves like the past: new stores, promotions or price changes are not known to it."
5,The data is synthetic; accuracy on real retail data would need to be re-measured.


In [9]:
conn.close()